# Quantization-Aware Training on Different Architectures

In [29]:
# Import necessary libraries for file handling, data manipulation, and visualization
import os
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Import libraries for working with images and transformations
from PIL import Image
import cv2 as cv

# Import PyTorch modules for model building, data handling, and evaluation
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torch.nn.functional as F
import torchvision.models as models
import torch.quantization as qt
from torch.quantization import QuantStub, DeQuantStub
from torch.utils.checkpoint import checkpoint
from torch.utils.data import Dataset, DataLoader, Subset
from timm import create_model

from torchinfo import summary

# Import libraries for machine learning metrics and model evaluation
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, r2_score, confusion_matrix
# import torchmetric
from tqdm import tqdm
from datetime import datetime
import json
import csv

import warnings
warnings.filterwarnings('ignore')
import gc

# Set the seed.
seed = 42
torch.manual_seed(seed)

In [30]:
data_dir="/home/sebastian-cruz6/cp-anemia-detection/data/cp-anemia/"
weights_dir="/home/sebastian-cruz6/cp-anemia-detection/notebooks/weights/"
metrics_dir="/home/sebastian-cruz6/cp-anemia-detection/notebooks/metrics/"

# data_dir = "/content/drive/MyDrive/CAWT_Sebastian_202425/CP-AnemiC/"
# weights_dir = "/content/drive/MyDrive/CAWT_Sebastian_202425/Weights/"
anemic_dir=data_dir+"/Anemic/"
non_anemic_dir=data_dir+"/Non-anemic/"
signature = "QUANTIZATION"

In [31]:
data_sheet_path = data_dir+"Anemia_Data_Collection_Sheet.csv"
data_sheet = pd.read_csv(data_sheet_path)
display(data_sheet)

,IMAGE_ID,HB_LEVEL,Severity,Age(Months),GENDER,REMARK,HOSPITAL,CITY/TOWN,MUNICIPALITY/DISTRICT,REGION,COUNTRY
0,Image_001,9.80,Moderate,6,Female,Anemic,Nkawie-Toase Government Hospital,Nkawie-Toase,Atwima Nwabiagya South,Ashanti,Ghana
1,Image_002,9.90,Moderate,24,Male,Anemic,Ejusu Government Hospital,Ejusu,Ejusu Municipality,Ashanti,Ghana
2,Image_003,11.10,Non-Anemic,24,Female,Non-anemic,Ahmadiyya Muslim Hospital,Tachiman,Techiman Municipality,Bono-East,Ghana
3,Image_004,12.50,Non-Anemic,12,Male,Non-anemic,Ahmadiyya Muslim Hospital,Tachiman,Techiman Municipality,Bono-East,Ghana
4,Image_005,9.90,Moderate,24,Male,Anemic,Sunyani Municipal Hospital,Sunyani,Sunyani Municipality,Bono,Ghana
...,...,...,...,...,...,...,...,...,...,...,...
705,Image_706,12.80,Non-Anemic,48,Male,Non-anemic,Bolgatanga Regional Hospital,Bolgatanga,Bolgatanga Municipality,Upper East,Ghana
706,Image_707,11.47,Non-Anemic,48,Female,Non-anemic,Ahmadiyya Muslim Hospital,Tachiman,Techiman Municipality,Bono-East,Ghana
707,Image_708,11.60,Non-Anemic,60,Male,Non-anemic,Komfo Anokye Teaching Hospital,Kumasi,Kumasi Metropolitan,Ashanti,Ghana
708,Image_709,12.10,Non-Anemic,48,Male,Non-anemic,Bolgatanga Regional Hospital,Bolgatanga,Bolgatanga Municipality,Upper East,Ghana


In [32]:
# Mapping diagnosis to severity
severity_mapping = {
    "Non-Anemic": 0,
    "Mild": 1,
    "Moderate": 2,
    "Severe": 3,
}

data_sheet['Severity'] = data_sheet['Severity'].map(severity_mapping)
display(data_sheet)

,IMAGE_ID,HB_LEVEL,Severity,Age(Months),GENDER,REMARK,HOSPITAL,CITY/TOWN,MUNICIPALITY/DISTRICT,REGION,COUNTRY
0,Image_001,9.80,2,6,Female,Anemic,Nkawie-Toase Government Hospital,Nkawie-Toase,Atwima Nwabiagya South,Ashanti,Ghana
1,Image_002,9.90,2,24,Male,Anemic,Ejusu Government Hospital,Ejusu,Ejusu Municipality,Ashanti,Ghana
2,Image_003,11.10,0,24,Female,Non-anemic,Ahmadiyya Muslim Hospital,Tachiman,Techiman Municipality,Bono-East,Ghana
3,Image_004,12.50,0,12,Male,Non-anemic,Ahmadiyya Muslim Hospital,Tachiman,Techiman Municipality,Bono-East,Ghana
4,Image_005,9.90,2,24,Male,Anemic,Sunyani Municipal Hospital,Sunyani,Sunyani Municipality,Bono,Ghana
...,...,...,...,...,...,...,...,...,...,...,...
705,Image_706,12.80,0,48,Male,Non-anemic,Bolgatanga Regional Hospital,Bolgatanga,Bolgatanga Municipality,Upper East,Ghana
706,Image_707,11.47,0,48,Female,Non-anemic,Ahmadiyya Muslim Hospital,Tachiman,Techiman Municipality,Bono-East,Ghana
707,Image_708,11.60,0,60,Male,Non-anemic,Komfo Anokye Teaching Hospital,Kumasi,Kumasi Metropolitan,Ashanti,Ghana
708,Image_709,12.10,0,48,Male,Non-anemic,Bolgatanga Regional Hospital,Bolgatanga,Bolgatanga Municipality,Upper East,Ghana


In [33]:
# Define data augmentations or transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=np.random.rand()),
    transforms.RandomVerticalFlip(p=np.random.rand()),
    transforms.RandomRotation(degrees=np.random.randint(0, 360)),
    transforms.RandomAffine(degrees=np.random.randint(0, 360)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Custom dataset class
class CPAnemiCDataset(Dataset):
    def __init__(self, dir, df, transform=None):
        self.dir = dir
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row['IMAGE_ID']
        img_folder = row['REMARK']
        img_path = os.path.join(self.dir, img_folder, img_id + ".png")
        img = Image.open(img_path).convert('RGB')

        if self.transform:
            img = self.transform(img)

        multiclass_label = torch.tensor(row['Severity'])
        hb_level = torch.tensor(row['HB_LEVEL'])

        return img, multiclass_label, hb_level

    # Load the dataset
image_dataset = CPAnemiCDataset(data_dir, data_sheet, transform=transform)
train_dataset, test_dataset = train_test_split(image_dataset, test_size=0.20, shuffle=True)

print(f"Image Dataset Size (All): {len(image_dataset)}, \
        Train Size: {len(train_dataset)}, \
        Test Size: {len(test_dataset)}")

BATCH_SIZE = 32
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

Image Dataset Size (All): 710,         Train Size: 568,         Test Size: 142


In [34]:
# Default device
device = torch.device('cpu')

# Check for CUDA availability
if torch.cuda.is_available():
    device = torch.device("cuda:1")
else:
    print("CUDA is not available, using CPU.")

print(f"Selected device: {device}")

Selected device: cuda:1


In [35]:
!nvidia-smi

Fri Mar 14 17:58:45 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.120                Driver Version: 550.120        CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        Off |   00000000:41:00.0 Off |                  Off |
| 30%   42C    P2            145W /  480W |    4405MiB /  24564MiB |     32%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
def get_model_size(model, model_type="pytorch", model_path="model.onnx"):
    """Returns model size in MB"""
    if model_type == "pytorch":
        torch.save(model.state_dict(), "tmp.pt")
        model_size = os.path.getsize("tmp.pt") / 1e6  # Convert bytes to MB
        os.remove("tmp.pt")
    elif model_type == "onnx":
        model_size = os.path.getsize(model_path) / 1e6
    return f"Model Size: {model_size:.2f} MB"

# Function to measure inference time & memory
def timed_forward(model, img, mode="pytorch", precision="fp32", ort_session=None, trt_context=None):
    """Measures inference time and memory usage for PyTorch, ONNX, and TensorRT."""
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    # Clear cache
    torch.cuda.empty_cache()
    gc.collect()

    # Record memory usage before inference
    torch.cuda.reset_peak_memory_stats()
    mem_before = torch.cuda.memory_allocated()
    max_mem_before = torch.cuda.max_memory_allocated()

    # Apply precision setting
    if precision == "fp16":
        model.half()
        img = img.half()

    # Start measuring latency
    start_event.record()

    if mode == "pytorch":
        class_pred, reg_pred = model(img)
    elif mode == "onnx":
        ort_inputs = {ort_session.get_inputs()[0].name: img.cpu().numpy()}
        output = ort_session.run(None, ort_inputs)
        class_pred, reg_pred = torch.tensor(output[0]), torch.tensor(output[1])

    elif mode == "trt":
        trt_context.execute_v2([img.contiguous().data_ptr()])
        class_pred, reg_pred = None, None  # Modify if using TensorRT output

    end_event.record()
    torch.cuda.synchronize()  # Ensure accurate timing
    latency = start_event.elapsed_time(end_event)  # Time in ms

    # Record memory usage after inference
    mem_after = torch.cuda.memory_allocated()
    max_mem_after = torch.cuda.max_memory_allocated()

    # Store stats
    stats = {
        "latency": latency,
        "malloc_before": mem_before,
        "malloc_after": mem_after,
        "max_malloc": max_mem_after,
    }

    return class_pred, reg_pred, stats

# Static Weighting Function. Set eta_class to desired importance (Classification > .5, Regression < .5, Equal == .5)
def sw_loss(loss_class, loss_reg, eta_class=0.5):
    eta_reg = 1 - eta_class
    total_loss = (eta_class * loss_class) + (eta_reg * loss_reg)
    return total_loss

In [37]:
class MultiModel(nn.Module):
    MODEL_MAPPING = {
        "mobilenetv2": lambda: models.mobilenet_v2(pretrained=False),
        "resnet18": lambda: models.resnet18(pretrained=False),
        "densenet121": lambda: models.densenet121(pretrained=False),
        "vgg16": lambda: models.vgg16(pretrained=False),
        "vit-tiny": lambda: create_model("vit_tiny_patch16_224", pretrained=False),
        "convnext-tiny": lambda: models.convnext_tiny(pretrained=False),
        "efficientnet-b0": lambda: models.efficientnet_b0(pretrained=False),
        "shufflenetv2-0.5x": lambda: models.shufflenet_v2_x0_5(pretrained=False),
        "regnety-400mf": lambda: models.regnet_y_400mf(pretrained=False),
        "mnasnet0_5": lambda: models.mnasnet0_5(pretrained=False),
        "ghostnetv2": lambda: create_model('ghostnetv2_100.in1k', pretrained=False),
        "tinynet-a": lambda: create_model("tinynet_a.in1k", pretrained=False)
    }

    FEATURE_LAYER_MAPPING = {
        "fc": ["resnet", "shufflenet", "regnet"],
        "classifier": ["densenet", "vgg", "mobilenet", "efficientnet",
                       "mnasnet","convnext", "ghostnet", "tinynet"],
        "head": ["vit"]
    }

    def __init__(self, model_name):
        super().__init__()

        self.quant = QuantStub() # Start quantization
        
        self.model_name = model_name.lower()

        if self.model_name not in self.MODEL_MAPPING:
            raise ValueError(f"Model {model_name} not supported")

        self.model = self.MODEL_MAPPING[self.model_name]()
        num_ftrs = self._get_feature_size()

        print(f"Initial Backbone {get_model_size(self.model)}")

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=0.2),
            nn.Linear(num_ftrs, 128),
            nn.ReLU(),
            nn.Linear(128, 5)
        )


        self._assign_classifier()
        self.dequant = DeQuantStub() # Convert back to fp32
        print(f"Modified Backbone {get_model_size(self.model)}\n")

    def _get_feature_size(self):
        """Retrieve the number of input features for the last layer."""
        # Special case for VGG16 since its features need flattening
        if "vgg" in self.model_name:
            return 25088  # VGG16 outputs (batch, 512, 7, 7) -> flattened to 25088

        feature_layers = {
            "fc": getattr(self.model, "fc", None),
            "classifier": getattr(self.model, "classifier", None),
            "head": getattr(self.model, "head", None)
        }

        for key, layer in feature_layers.items():
            if layer:
                return layer[-1].in_features if isinstance(layer, nn.Sequential) else layer.in_features

        return getattr(self.model, "num_features", None)

    def _assign_classifier(self):
        """Assigns the appropriate classifier to the model based on its architecture."""
        if "vgg" in self.model_name:
            self.model.classifier = self.classifier
        else:
          for attr, models in self.FEATURE_LAYER_MAPPING.items():
            if any(m in self.model_name for m in models):
                setattr(self.model, attr, self.classifier)
                return

    def forward(self, x):
        output = self.model(x)
        return output[:, :4], output[:, 4]  # Class probabilities and Hb level estimate

In [38]:
models_list = ["mobilenetv2", "resnet18", "densenet121", "vgg16", "vit-tiny",
               "efficientnet-b0", "shufflenetv2-0.5x", "regnety-400mf",
               "mnasnet0_5", "convnext-tiny", "ghostnetv2", "tinynet-a"
               ]
for arch in models_list:
    print(f"Loading model: {arch}")
    model = MultiModel(arch).to(device)
    print(summary(model))
    # print(model)

Loading model: mobilenetv2
Initial Backbone Model Size: 14.24 MB
Modified Backbone Model Size: 9.78 MB

Layer (type:depth-idx)                                  Param #
MultiModel                                              --
├─QuantStub: 1-1                                        --
├─MobileNetV2: 1-2                                      --
│    └─Sequential: 2-1                                  --
│    │    └─Conv2dNormActivation: 3-1                   928
│    │    └─InvertedResidual: 3-2                       896
│    │    └─InvertedResidual: 3-3                       5,136
│    │    └─InvertedResidual: 3-4                       8,832
│    │    └─InvertedResidual: 3-5                       10,000
│    │    └─InvertedResidual: 3-6                       14,848
│    │    └─InvertedResidual: 3-7                       14,848
│    │    └─InvertedResidual: 3-8                       21,056
│    │    └─InvertedResidual: 3-9                       54,272
│    │    └─InvertedResidual: 3-10   

In [39]:
def train(dataloader, model, class_loss, reg1_loss, reg2_loss, optimizer):
    """Trains the model and logs additional metrics."""
    model.train()
    model.qconfig = torch.quantization.get_default_qat_qconfig('fbgemm')  # Choose backend. FBGEMM is Best for CPUs (like Intel, ARM).
    model = torch.quantization.prepare_qat(model)

    total_loss = 0
    total_ce_loss = 0
    total_mse_loss = 0
    total_mae_loss = 0
    correct = 0
    total_samples = 0

    all_preds = []
    all_targets = []
    all_probs = []
    all_hb_targets = []
    all_hb_preds = []

    for _, (img, multiclass, hb_level) in enumerate(dataloader):
        img = img.to(device)
        multiclass = multiclass.to(device).long()
        hb_level = hb_level.to(device).unsqueeze(1).float()

        optimizer.zero_grad()

        # Forward pass
        class_pred, reg_pred = model(img)

        # Compute losses
        ce_loss = class_loss(class_pred, multiclass)
        mse_loss = reg1_loss(reg_pred, hb_level)
        mae_loss = reg2_loss(reg_pred, hb_level)
        loss = sw_loss(ce_loss, mse_loss, 0.7)  # Weighted loss

        # Backpropagation
        loss.backward()
        optimizer.step()

        # Track total losses
        total_loss += loss.item()
        total_ce_loss += ce_loss.item()
        total_mse_loss += mse_loss.item()
        total_mae_loss += mae_loss.item()

        # Compute classification accuracy
        class_probs = F.softmax(class_pred, dim=1)
        highest_prob_class = torch.argmax(class_probs, dim=1)

        correct += (highest_prob_class == multiclass).sum().item()
        total_samples += multiclass.size(0)

        # Collect data for additional metrics
        all_preds.extend(highest_prob_class.detach().cpu().numpy())
        all_targets.extend(multiclass.detach().cpu().numpy())
        all_probs.extend(class_probs.detach().cpu().numpy())
        all_hb_targets.extend(hb_level.detach().cpu().numpy())
        all_hb_preds.extend(reg_pred.squeeze().cpu().detach().numpy())

    # Compute additional metrics
    precision = precision_score(all_targets, all_preds, average="weighted")
    recall = recall_score(all_targets, all_preds, average="weighted")
    f1 = f1_score(all_targets, all_preds, average="weighted")
    auc = roc_auc_score(all_targets, all_probs, multi_class="ovr")
    r2 = r2_score(all_hb_targets, all_hb_preds)

    # Compute final statistics
    avg_loss = total_loss / len(dataloader)
    avg_ce_loss = total_ce_loss / len(dataloader)
    avg_mse_loss = total_mse_loss / len(dataloader)
    avg_mae_loss = total_mae_loss / len(dataloader)
    accuracy = correct / total_samples

    # Store metrics
    final_metrics = [avg_loss, avg_ce_loss, accuracy, precision, recall, f1, auc, r2, avg_mae_loss, avg_mse_loss]

    return final_metrics


In [40]:
def eval(dataloader, model, class_loss, reg1_loss, reg2_loss, mode="pytorch", precision="fp32", ort_session=None, trt_context=None):
    """Evaluates the model with additional metrics: Precision, Recall, AUC, F1, R², Memory Usage, and Latency."""
    model.eval()
    if precision == "int8":
        model = torch.quantization.convert(model).to(device)
    
    mean_stats = []
    total_loss = 0
    total_ce_loss = 0
    total_mse_loss = 0
    total_mae_loss = 0
    correct = 0
    total_samples = 0

    all_preds = []
    all_targets = []
    all_probs = []
    all_hb_targets = []
    all_hb_preds = []

    torch.cuda.empty_cache()
    gc.collect()

    with torch.no_grad():
        for _, (img, multiclass, hb_level) in enumerate(dataloader):
            img = img.to(device)
            multiclass = multiclass.to(device).long()
            hb_level = hb_level.to(device).unsqueeze(1).float()

            # Forward pass with latency & memory tracking
            class_pred, reg_pred, stats = timed_forward(model, img, mode, ort_session, trt_context)
            mean_stats.append(stats)

            # Compute losses
            ce_loss = class_loss(class_pred, multiclass)
            mse_loss = reg1_loss(reg_pred, hb_level)
            mae_loss = reg2_loss(reg_pred, hb_level)
            loss = sw_loss(ce_loss, mse_loss, 0.7)

            # Track total losses
            total_loss += loss.item()
            total_ce_loss += ce_loss.item()
            total_mse_loss += mse_loss.item()
            total_mae_loss += mae_loss.item()

            # Compute classification accuracy
            class_probs = F.softmax(class_pred, dim=1)
            highest_prob_class = torch.argmax(class_probs, dim=1)

            correct += (highest_prob_class == multiclass).sum().item()
            total_samples += multiclass.size(0)

            # Collect data for additional metrics
            all_preds.extend(highest_prob_class.detach().cpu().numpy())
            all_targets.extend(multiclass.detach().cpu().numpy())
            all_probs.extend(class_probs.detach().cpu().numpy())
            all_hb_targets.extend(hb_level.detach().cpu().numpy())
            all_hb_preds.extend(reg_pred.squeeze().detach().cpu().numpy())

    # Compute mean statistics
    mean_latency = np.mean([s["latency"] for s in mean_stats])
    mean_mem_before = np.mean([s["malloc_before"] for s in mean_stats]) / 1_048_576  # Convert bytes to MB
    mean_mem_after = np.mean([s["malloc_after"] for s in mean_stats]) / 1_048_576  # Convert bytes to MB
    mean_max_mem = np.mean([s["max_malloc"] for s in mean_stats]) / 1_048_576  # Convert bytes to MB

    # Store final mean statistics
    final_mean_stats = [mean_latency, mean_mem_before, mean_mem_after, mean_max_mem]

    # Compute additional evaluation metrics
    precision = precision_score(all_targets, all_preds, average="weighted")
    recall = recall_score(all_targets, all_preds, average="weighted")
    f1 = f1_score(all_targets, all_preds, average="weighted")
    auc = roc_auc_score(all_targets, all_probs, multi_class="ovr")
    r2 = r2_score(all_hb_targets, all_hb_preds)

    # Compute confusion matrix
    # cm = confusion_matrix(all_targets, all_preds)

    # Compute final average losses
    avg_loss = total_loss / len(dataloader)
    avg_ce_loss = total_ce_loss / len(dataloader)
    avg_mse_loss = total_mse_loss / len(dataloader)
    avg_mae_loss = total_mae_loss / len(dataloader)
    accuracy = correct / total_samples

    # Store metrics
    final_metrics = [avg_loss, avg_ce_loss, accuracy, precision, recall, f1, auc, r2, avg_mae_loss, avg_mse_loss]

    return final_metrics, final_mean_stats

In [41]:
def main(ARCH, BATCH_SIZE=32, EPOCHS=150, FOLDS=5):

    # Define loss functions
    cross_entropy_loss = torch.nn.CrossEntropyLoss()  # Multi-class classification loss
    mse_loss = torch.nn.MSELoss()  # Regression loss
    mae_loss = torch.nn.L1Loss()  # Regression loss

    # Set up 5-Fold Cross Validation
    kf = KFold(n_splits=FOLDS, shuffle=True, random_state=42)

    print("=" * 100)
    print(f"Training Model: {ARCH}")

    # === INITIALIZE MODEL ===
    model = MultiModel(ARCH).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    best_val_acc = -float("inf")  # Track best validation accuracy
    train_metrics_list = []
    val_metrics_list = []

    # === TRAINING LOOP ===
    for epoch in range(EPOCHS):
        print(f"\nEpoch {epoch+1}/{EPOCHS}")
        fold = 1

        for train_idx, val_idx in kf.split(range(len(image_dataset))):
            train_subset = Subset(image_dataset, train_idx)
            val_subset = Subset(image_dataset, val_idx)

            train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
            val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

            if fold == FOLDS:
                # === VALIDATION PHASE ===
                val_metrics, val_stats = eval(val_loader, model, cross_entropy_loss, mse_loss, mae_loss)
                print(
                    f"Validation: Fold {fold} - Total Loss: {val_metrics[0]:.4f}, Cross Entropy: {val_metrics[1]:4f}, Accuracy: {val_metrics[2]:.4f}, "
                    f"Precision: {val_metrics[3]:.4f}, Recall: {val_metrics[4]:.4f}, F1 Score: {val_metrics[5]:.4f}, AUC: {val_metrics[6]:.4f}, "
                    f"R2 Score: {val_metrics[7]:4f}, MAE: {val_metrics[8]:.4f}, MSE: {val_metrics[9]:.4f}"
                )
                print(
                    f"Avg Latency (ms): {val_stats[0]:.2f}, Avg Memory Before (MB): {val_stats[1]:.2f}, "
                    f"Avg Memory After (MB): {val_stats[2]:.2f}, Avg Max Memory (MB): {val_stats[3]:.2f}"
                )

                # Save best model based on validation accuracy
                if val_metrics[2] > best_val_acc:
                    best_val_acc = val_metrics[2]
                    torch.save(
                        model.state_dict(),
                        f"{weights_dir}/pytorch/model_best_accuracy_{ARCH}_{signature}.pth",
                    )
                    print(f"Best model saved with Accuracy: {best_val_acc:.4f}")

                # Store validation metrics
                val_metrics_list.append(
                    {
                        "epoch": epoch + 1,
                        "fold": fold,
                        "total_loss": val_metrics[0],
                        "cross_entropy_loss": val_metrics[1],
                        "accuracy": val_metrics[2],
                        "precision": val_metrics[3],
                        "recall": val_metrics[4],
                        "f1_score": val_metrics[5],
                        "auc": val_metrics[6],
                        "r2_score": val_metrics[7],
                        "mae_loss": val_metrics[8],
                        "mse_loss": val_metrics[9],
                        "latency": val_stats[0],
                        "malloc_before": val_stats[1],
                        "malloc_after": val_stats[2],
                        "max_malloc": val_stats[3],
                    }
                )

            else:
                # === TRAINING PHASE ===
                train_metrics = train(train_loader, model, cross_entropy_loss, mse_loss, mae_loss, optimizer)
                print(
                    f"Training: Fold {fold} - Total Loss: {train_metrics[0]:.4f}, Cross Entropy: {train_metrics[1]:4f}, Accuracy: {train_metrics[2]:.4f}, "
                    f"Precision: {train_metrics[3]:.4f}, Recall: {train_metrics[4]:.4f}, F1 Score: {train_metrics[5]:.4f}, AUC: {train_metrics[6]:.4f}, "
                    f"R2 Score: {train_metrics[7]:4f}, MAE: {train_metrics[8]:.4f}, MSE: {train_metrics[9]:.4f}"
                )

                # Store training metrics
                train_metrics_list.append(
                    {
                        "epoch": epoch + 1,
                        "fold": fold,
                        "total_loss": train_metrics[0],
                        "cross_entropy_loss": train_metrics[1],
                        "accuracy": train_metrics[2],
                        "precision": train_metrics[3],
                        "recall": train_metrics[4],
                        "f1_score": train_metrics[5],
                        "auc": train_metrics[6],
                        "r2_score": train_metrics[7],
                        "mae_loss": train_metrics[8],
                        "mse_loss": train_metrics[9],
                    }
                )

            fold += 1  # Move to next fold
        
        keys = train_metrics_list[0].keys()
        with open(f"{metrics_dir}/pytorch/training_metrics_{ARCH}_{signature}.csv", 'w', newline='') as output_file:
            dict_writer = csv.DictWriter(output_file, keys)
            dict_writer.writeheader()
            dict_writer.writerows(train_metrics_list)

        keys = val_metrics_list[0].keys()
        with open(f"{metrics_dir}/pytorch/validation_metrics_{ARCH}_{signature}.csv", 'w', newline='') as output_file:
            dict_writer = csv.DictWriter(output_file, keys)
            dict_writer.writeheader()
            dict_writer.writerows(val_metrics_list)

    print(f"\nFine-tuned {get_model_size(model)}")
    print("=" * 100)

In [ ]:
import onnxruntime as ort

def infer(arch, path, mode="pytorch"):
  
    # Define loss functions
    cross_entropy_loss = torch.nn.CrossEntropyLoss()  # Multi-class classification loss
    mse_loss = torch.nn.MSELoss()  # Regression loss
    mae_loss = torch.nn.L1Loss()  # Regression loss

    test_metrics_list = []
    print("="*100)
    print(f"{arch}")

    model = MultiModel(arch).to(device)
    model.load_state_dict(torch.load(f"{weights_dir}pytorch/model_best_accuracy_{arch}_{path}.pth"))
    print(f"Testing {arch} in {mode} mode")
    
    for precision in ["fp32", "fp16", "int8"]:
        print(f"Running inference for {precision}")

        ort_session = None

        if mode == "onnx":
            dummy_input = torch.randn(1, 3, 224, 224).to(device)
            onnx_path = f"{weights_dir}onnx/model_best_accuracy_{arch}_{path}.onnx"
            
            try:
                # Export ONNX Model
                torch.onnx.export(model, dummy_input, onnx_path, opset_version=11)
                print(f"Successfully exported ONNX model: {onnx_path}")

                # Load ONNX Model
                ort_session = ort.InferenceSession(onnx_path, providers=["CUDAExecutionProvider"])
            except Exception as e:
                print(f"ONNX Export/Loading Failed: {e}")
                ort_session = None  # Avoid passing None session to eval()

        # === Testing PHASE ===
        test_metrics, test_stats = eval(test_loader, model, cross_entropy_loss, mse_loss, mae_loss, mode, precision, ort_session)
        print(
            f"Testing: Total Loss: {test_metrics[0]:.4f}, Cross Entropy: {test_metrics[1]:4f}, Accuracy: {test_metrics[2]:.4f}, "
            f"Precision: {test_metrics[3]:.4f}, Recall: {test_metrics[4]:.4f}, F1 Score: {test_metrics[5]:.4f}, AUC: {test_metrics[6]:.4f}, "
            f"R2 Score: {test_metrics[7]:4f}, MAE: {test_metrics[8]:.4f}, MSE: {test_metrics[9]:.4f}"
        )
        print(
            f"Avg Latency (ms): {test_stats[0]:.2f}, Avg Memory Before (MB): {test_stats[1]:.2f}, "
            f"Avg Memory After (MB): {test_stats[2]:.2f}, Avg Max Memory (MB): {test_stats[3]:.2f}"
        )

        # Store validation metrics
        test_metrics_list.append(
            {
                "total_loss": test_metrics[0],
                "cross_entropy_loss": test_metrics[1],
                "accuracy": test_metrics[2],
                "precision": test_metrics[3],
                "recall": test_metrics[4],
                "f1_score": test_metrics[5],
                "auc": test_metrics[6],
                "r2_score": test_metrics[7],
                "mae_loss": test_metrics[8],
                "mse_loss": test_metrics[9],
                "latency": test_stats[0],
                "malloc_before": test_stats[1],
                "malloc_after": test_stats[2],
                "max_malloc": test_stats[3],
            }
        )

        keys = test_metrics_list[0].keys()
        with open(f"{metrics_dir}/pytorch/testing_metrics_{arch}_{signature}.csv", 'w', newline='') as output_file:
            dict_writer = csv.DictWriter(output_file, keys)
            dict_writer.writeheader()
            dict_writer.writerows(test_metrics_list)

In [103]:
main("mobilenetv2", EPOCHS=10)

Training Model: mobilenetv2
Initial Backbone Model Size: 14.24 MB
Modified Backbone Model Size: 9.78 MB


Epoch 1/10
Training: Fold 1 - Total Loss: 34.3929, Cross Entropy: 1.333070, Accuracy: 0.4085, Precision: 0.2772, Recall: 0.4085, F1 Score: 0.2496, AUC: 0.4883, R2 Score: -20.373997, MAE: 10.3092, MSE: 111.5324
Training: Fold 2 - Total Loss: 34.4830, Cross Entropy: 1.335900, Accuracy: 0.3926, Precision: 0.2224, Recall: 0.3926, F1 Score: 0.2369, AUC: 0.4995, R2 Score: -21.559852, MAE: 10.3385, MSE: 111.8261
Training: Fold 3 - Total Loss: 34.3036, Cross Entropy: 1.342106, Accuracy: 0.3732, Precision: 0.2532, Recall: 0.3732, F1 Score: 0.2204, AUC: 0.4973, R2 Score: -21.912936, MAE: 10.3106, MSE: 111.2136
Training: Fold 4 - Total Loss: 34.8987, Cross Entropy: 1.334172, Accuracy: 0.4225, Precision: 0.3818, Recall: 0.4225, F1 Score: 0.2601, AUC: 0.5112, R2 Score: -20.368900, MAE: 10.3848, MSE: 113.2160
Validation: Fold 5 - Total Loss: 35.1876, Cross Entropy: 1.362463, Accuracy: 0.3873, Pr

In [104]:
main("resnet18", EPOCHS=10)

Training Model: resnet18
Initial Backbone Model Size: 46.83 MB
Modified Backbone Model Size: 45.04 MB


Epoch 1/10
Training: Fold 1 - Total Loss: 33.4075, Cross Entropy: 1.370543, Accuracy: 0.3944, Precision: 0.1781, Recall: 0.3944, F1 Score: 0.2438, AUC: 0.5408, R2 Score: -19.678282, MAE: 10.1466, MSE: 108.1605
Training: Fold 2 - Total Loss: 33.3319, Cross Entropy: 1.383091, Accuracy: 0.3732, Precision: 0.1663, Recall: 0.3732, F1 Score: 0.2296, AUC: 0.4952, R2 Score: -20.818493, MAE: 10.1445, MSE: 107.8791
Training: Fold 3 - Total Loss: 33.2922, Cross Entropy: 1.386819, Accuracy: 0.3504, Precision: 0.1490, Recall: 0.3504, F1 Score: 0.2083, AUC: 0.5152, R2 Score: -21.177077, MAE: 10.1432, MSE: 107.7379
Training: Fold 4 - Total Loss: 33.9469, Cross Entropy: 1.371180, Accuracy: 0.3944, Precision: 0.5892, Recall: 0.3944, F1 Score: 0.2520, AUC: 0.5293, R2 Score: -19.678488, MAE: 10.2297, MSE: 109.9570
Validation: Fold 5 - Total Loss: 34.4076, Cross Entropy: 1.340247, Accuracy: 0.3873, Prec

In [105]:
main("densenet121", EPOCHS=10)

Training Model: densenet121
Initial Backbone Model Size: 32.47 MB
Modified Backbone Model Size: 28.90 MB


Epoch 1/10
Training: Fold 1 - Total Loss: 33.2751, Cross Entropy: 1.397721, Accuracy: 0.2306, Precision: 0.2947, Recall: 0.2306, F1 Score: 0.2353, AUC: 0.4715, R2 Score: -19.553164, MAE: 10.1214, MSE: 107.6557
Training: Fold 2 - Total Loss: 33.2779, Cross Entropy: 1.397159, Accuracy: 0.1989, Precision: 0.2664, Recall: 0.1989, F1 Score: 0.2059, AUC: 0.4909, R2 Score: -20.709124, MAE: 10.1342, MSE: 107.6662
Training: Fold 3 - Total Loss: 33.1490, Cross Entropy: 1.393777, Accuracy: 0.2271, Precision: 0.2949, Recall: 0.2271, F1 Score: 0.2376, AUC: 0.4923, R2 Score: -21.030192, MAE: 10.1202, MSE: 107.2446
Training: Fold 4 - Total Loss: 33.7163, Cross Entropy: 1.395832, Accuracy: 0.2148, Precision: 0.2715, Recall: 0.2148, F1 Score: 0.2236, AUC: 0.4766, R2 Score: -19.565458, MAE: 10.1899, MSE: 109.1306
Validation: Fold 5 - Total Loss: 35.0686, Cross Entropy: 1.474448, Accuracy: 0.2324, P

In [106]:
main("vgg16", EPOCHS=10)

Training Model: vgg16
Initial Backbone Model Size: 553.44 MB
Modified Backbone Model Size: 71.72 MB


Epoch 1/10
Training: Fold 1 - Total Loss: 34.4692, Cross Entropy: 1.401370, Accuracy: 0.2077, Precision: 0.2247, Recall: 0.2077, F1 Score: 0.2001, AUC: 0.4952, R2 Score: -20.339274, MAE: 10.3127, MSE: 111.6275
Training: Fold 2 - Total Loss: 34.4082, Cross Entropy: 1.402654, Accuracy: 0.2183, Precision: 0.2269, Recall: 0.2183, F1 Score: 0.2020, AUC: 0.5112, R2 Score: -21.523038, MAE: 10.3179, MSE: 111.4211
Training: Fold 3 - Total Loss: 34.2718, Cross Entropy: 1.402558, Accuracy: 0.2042, Precision: 0.2309, Recall: 0.2042, F1 Score: 0.1991, AUC: 0.5129, R2 Score: -21.872463, MAE: 10.2998, MSE: 110.9666
Training: Fold 4 - Total Loss: 34.9818, Cross Entropy: 1.398391, Accuracy: 0.2007, Precision: 0.2349, Recall: 0.2007, F1 Score: 0.2021, AUC: 0.4966, R2 Score: -20.342561, MAE: 10.3932, MSE: 113.3430
Validation: Fold 5 - Total Loss: 35.6510, Cross Entropy: 1.396611, Accuracy: 0.1549, Precis

In [107]:
main("vit-tiny", EPOCHS=10)

Training Model: vit-tiny
Initial Backbone Model Size: 22.92 MB
Modified Backbone Model Size: 22.25 MB


Epoch 1/10
Training: Fold 1 - Total Loss: 34.3157, Cross Entropy: 1.368458, Accuracy: 0.3592, Precision: 0.2631, Recall: 0.3592, F1 Score: 0.2427, AUC: 0.4893, R2 Score: -20.255673, MAE: 10.2915, MSE: 111.1927
Training: Fold 2 - Total Loss: 34.3270, Cross Entropy: 1.375246, Accuracy: 0.3468, Precision: 0.2986, Recall: 0.3468, F1 Score: 0.2524, AUC: 0.5020, R2 Score: -21.416105, MAE: 10.3077, MSE: 111.2144
Training: Fold 3 - Total Loss: 33.9957, Cross Entropy: 1.382014, Accuracy: 0.3151, Precision: 0.2077, Recall: 0.3151, F1 Score: 0.2090, AUC: 0.5167, R2 Score: -21.751221, MAE: 10.2566, MSE: 110.0943
Training: Fold 4 - Total Loss: 34.7689, Cross Entropy: 1.370891, Accuracy: 0.3592, Precision: 0.2298, Recall: 0.3592, F1 Score: 0.2440, AUC: 0.4852, R2 Score: -20.274567, MAE: 10.3618, MSE: 112.6976
Validation: Fold 5 - Total Loss: 35.5722, Cross Entropy: 1.376259, Accuracy: 0.3521, Prec

In [108]:
main("efficientnet-b0", EPOCHS=10)

Training Model: efficientnet-b0
Initial Backbone Model Size: 21.43 MB
Modified Backbone Model Size: 16.96 MB


Epoch 1/10
Training: Fold 1 - Total Loss: 34.2308, Cross Entropy: 1.403117, Accuracy: 0.1074, Precision: 0.3179, Recall: 0.1074, F1 Score: 0.0840, AUC: 0.5331, R2 Score: -20.234610, MAE: 10.2756, MSE: 110.8288
Training: Fold 2 - Total Loss: 34.3930, Cross Entropy: 1.408473, Accuracy: 0.1074, Precision: 0.2225, Recall: 0.1074, F1 Score: 0.0911, AUC: 0.4831, R2 Score: -21.424582, MAE: 10.3158, MSE: 111.3568
Training: Fold 3 - Total Loss: 34.1198, Cross Entropy: 1.407530, Accuracy: 0.1144, Precision: 0.1839, Recall: 0.1144, F1 Score: 0.0988, AUC: 0.4793, R2 Score: -21.763745, MAE: 10.2756, MSE: 110.4486
Training: Fold 4 - Total Loss: 34.8164, Cross Entropy: 1.404791, Accuracy: 0.1039, Precision: 0.1940, Recall: 0.1039, F1 Score: 0.0718, AUC: 0.5191, R2 Score: -20.239864, MAE: 10.3658, MSE: 112.7769
Validation: Fold 5 - Total Loss: 35.8699, Cross Entropy: 1.395128, Accuracy: 0.070

In [109]:
main("shufflenetv2-0.5x", EPOCHS=10)

Training Model: shufflenetv2-0.5x
Initial Backbone Model Size: 5.59 MB
Modified Backbone Model Size: 2.02 MB


Epoch 1/10
Training: Fold 1 - Total Loss: 34.5323, Cross Entropy: 1.381098, Accuracy: 0.3046, Precision: 0.2863, Recall: 0.3046, F1 Score: 0.2473, AUC: 0.4923, R2 Score: -20.359194, MAE: 10.3268, MSE: 111.8852
Training: Fold 2 - Total Loss: 34.5393, Cross Entropy: 1.377427, Accuracy: 0.3116, Precision: 0.3147, Recall: 0.3116, F1 Score: 0.2562, AUC: 0.5237, R2 Score: -21.571573, MAE: 10.3422, MSE: 111.9169
Training: Fold 3 - Total Loss: 34.3233, Cross Entropy: 1.378812, Accuracy: 0.3134, Precision: 0.3164, Recall: 0.3134, F1 Score: 0.2578, AUC: 0.5254, R2 Score: -21.895705, MAE: 10.3124, MSE: 111.1938
Training: Fold 4 - Total Loss: 34.9941, Cross Entropy: 1.380392, Accuracy: 0.3011, Precision: 0.3427, Recall: 0.3011, F1 Score: 0.2573, AUC: 0.5153, R2 Score: -20.354956, MAE: 10.3980, MSE: 113.4261
Validation: Fold 5 - Total Loss: 35.9724, Cross Entropy: 1.385146, Accuracy: 0.309

In [110]:
main("regnety-400mf", EPOCHS=10)

Training Model: regnety-400mf
Initial Backbone Model Size: 17.61 MB
Modified Backbone Model Size: 16.07 MB


Epoch 1/10
Training: Fold 1 - Total Loss: 34.4724, Cross Entropy: 1.485133, Accuracy: 0.1479, Precision: 0.5638, Recall: 0.1479, F1 Score: 0.1346, AUC: 0.4943, R2 Score: -20.289174, MAE: 10.3056, MSE: 111.4427
Training: Fold 2 - Total Loss: 34.3571, Cross Entropy: 1.484697, Accuracy: 0.1426, Precision: 0.3519, Recall: 0.1426, F1 Score: 0.1315, AUC: 0.4818, R2 Score: -21.420976, MAE: 10.2977, MSE: 111.0592
Training: Fold 3 - Total Loss: 34.2484, Cross Entropy: 1.477733, Accuracy: 0.1532, Precision: 0.2818, Recall: 0.1532, F1 Score: 0.1561, AUC: 0.4783, R2 Score: -21.813968, MAE: 10.2873, MSE: 110.7133
Training: Fold 4 - Total Loss: 34.8961, Cross Entropy: 1.478225, Accuracy: 0.1391, Precision: 0.4052, Recall: 0.1391, F1 Score: 0.1275, AUC: 0.5089, R2 Score: -20.257329, MAE: 10.3692, MSE: 112.8711
Validation: Fold 5 - Total Loss: 36.2003, Cross Entropy: 1.415249, Accuracy: 0.0704,

In [111]:
main("mnasnet0_5", EPOCHS=10)

Training Model: mnasnet0_5
Initial Backbone Model Size: 9.04 MB
Modified Backbone Model Size: 4.58 MB


Epoch 1/10
Training: Fold 1 - Total Loss: 34.3416, Cross Entropy: 1.405684, Accuracy: 0.1320, Precision: 0.4211, Recall: 0.1320, F1 Score: 0.1184, AUC: 0.5106, R2 Score: -20.308167, MAE: 10.2919, MSE: 111.1921
Training: Fold 2 - Total Loss: 34.4809, Cross Entropy: 1.409796, Accuracy: 0.1215, Precision: 0.3710, Recall: 0.1215, F1 Score: 0.1111, AUC: 0.4766, R2 Score: -21.486518, MAE: 10.3294, MSE: 111.6467
Training: Fold 3 - Total Loss: 34.3768, Cross Entropy: 1.402607, Accuracy: 0.1250, Precision: 0.3304, Recall: 0.1250, F1 Score: 0.1175, AUC: 0.5284, R2 Score: -21.830960, MAE: 10.3174, MSE: 111.3167
Training: Fold 4 - Total Loss: 35.0252, Cross Entropy: 1.409397, Accuracy: 0.1162, Precision: 0.2982, Recall: 0.1162, F1 Score: 0.0979, AUC: 0.4915, R2 Score: -20.316938, MAE: 10.3994, MSE: 113.4619
Validation: Fold 5 - Total Loss: 35.9507, Cross Entropy: 1.383586, Accuracy: 0.3099, Prec

In [112]:
main("convnext-tiny", EPOCHS=10)

Training Model: convnext-tiny
Initial Backbone Model Size: 114.41 MB
Modified Backbone Model Size: 111.72 MB


Epoch 1/10
Training: Fold 1 - Total Loss: 34.7542, Cross Entropy: 1.386022, Accuracy: 0.2570, Precision: 0.2882, Recall: 0.2570, F1 Score: 0.2318, AUC: 0.5189, R2 Score: -20.516698, MAE: 10.3645, MSE: 112.6132
Training: Fold 2 - Total Loss: 34.7799, Cross Entropy: 1.384147, Accuracy: 0.2588, Precision: 0.4719, Recall: 0.2588, F1 Score: 0.2297, AUC: 0.5431, R2 Score: -21.722571, MAE: 10.3815, MSE: 112.7032
Training: Fold 3 - Total Loss: 34.6044, Cross Entropy: 1.394068, Accuracy: 0.2676, Precision: 0.3130, Recall: 0.2676, F1 Score: 0.2408, AUC: 0.5006, R2 Score: -22.086385, MAE: 10.3558, MSE: 112.0951
Training: Fold 4 - Total Loss: 35.2687, Cross Entropy: 1.385615, Accuracy: 0.2782, Precision: 0.2823, Recall: 0.2782, F1 Score: 0.2445, AUC: 0.5109, R2 Score: -20.548263, MAE: 10.4423, MSE: 114.3293
Validation: Fold 5 - Total Loss: 35.8978, Cross Entropy: 1.387232, Accuracy: 0.345

In [113]:
main("ghostnetv2", EPOCHS=10)

Training Model: ghostnetv2
Initial Backbone Model Size: 25.11 MB
Modified Backbone Model Size: 20.64 MB


Epoch 1/10
Training: Fold 1 - Total Loss: 34.8078, Cross Entropy: 1.380410, Accuracy: 0.3539, Precision: 0.3358, Recall: 0.3539, F1 Score: 0.3089, AUC: 0.4988, R2 Score: -20.622445, MAE: 10.3726, MSE: 112.8051
Training: Fold 2 - Total Loss: 34.9232, Cross Entropy: 1.378916, Accuracy: 0.3539, Precision: 0.3436, Recall: 0.3539, F1 Score: 0.3193, AUC: 0.5571, R2 Score: -21.816727, MAE: 10.4039, MSE: 113.1931
Training: Fold 3 - Total Loss: 34.7265, Cross Entropy: 1.383468, Accuracy: 0.3187, Precision: 0.3056, Recall: 0.3187, F1 Score: 0.2806, AUC: 0.4903, R2 Score: -22.174370, MAE: 10.3748, MSE: 112.5269
Training: Fold 4 - Total Loss: 35.3803, Cross Entropy: 1.379614, Accuracy: 0.3415, Precision: 0.3484, Recall: 0.3415, F1 Score: 0.3214, AUC: 0.5099, R2 Score: -20.606051, MAE: 10.4612, MSE: 114.7152
Validation: Fold 5 - Total Loss: 35.5619, Cross Entropy: 1.385955, Accuracy: 0.2324, Pr

In [114]:
main("tinynet-a", EPOCHS=10)

Training Model: tinynet-a
Initial Backbone Model Size: 25.08 MB
Modified Backbone Model Size: 20.62 MB


Epoch 1/10
Training: Fold 1 - Total Loss: 34.2180, Cross Entropy: 1.376339, Accuracy: 0.3257, Precision: 0.2952, Recall: 0.3257, F1 Score: 0.3058, AUC: 0.5414, R2 Score: -20.146265, MAE: 10.2775, MSE: 110.8484
Training: Fold 2 - Total Loss: 34.1973, Cross Entropy: 1.382716, Accuracy: 0.2746, Precision: 0.2767, Recall: 0.2746, F1 Score: 0.2727, AUC: 0.4932, R2 Score: -21.312716, MAE: 10.2875, MSE: 110.7646
Training: Fold 3 - Total Loss: 34.0297, Cross Entropy: 1.384983, Accuracy: 0.2852, Precision: 0.2517, Recall: 0.2852, F1 Score: 0.2630, AUC: 0.5067, R2 Score: -21.660836, MAE: 10.2639, MSE: 110.2008
Training: Fold 4 - Total Loss: 34.7113, Cross Entropy: 1.382419, Accuracy: 0.3099, Precision: 0.2525, Recall: 0.3099, F1 Score: 0.2717, AUC: 0.4858, R2 Score: -20.144008, MAE: 10.3532, MSE: 112.4786
Validation: Fold 5 - Total Loss: 35.6661, Cross Entropy: 1.368784, Accuracy: 0.3873, Pre

In [19]:
for arch in models_list:
    infer(arch, "QUANTIZATION")

mobilenetv2
Initial Backbone Model Size: 14.24 MB
Modified Backbone Model Size: 9.78 MB

Testing: Total Loss: 33.5098, Cross Entropy: 1.362259, Accuracy: 0.3662, Precision: 0.1341, Recall: 0.3662, F1 Score: 0.1963, AUC: 0.5000, R2 Score: -20.658460, MAE: 10.1787, MSE: 108.5206
Avg Latency (ms): 3.41, Avg Memory Before (MB): 0.00, Avg Memory After (MB): 0.00, Avg Max Memory (MB): 0.00
resnet18
Initial Backbone Model Size: 46.83 MB
Modified Backbone Model Size: 45.04 MB

Testing: Total Loss: 32.7224, Cross Entropy: 1.352546, Accuracy: 0.3662, Precision: 0.1341, Recall: 0.3662, F1 Score: 0.1963, AUC: 0.4795, R2 Score: -20.137611, MAE: 10.0501, MSE: 105.9186
Avg Latency (ms): 2.08, Avg Memory Before (MB): 0.00, Avg Memory After (MB): 0.00, Avg Max Memory (MB): 0.00
densenet121
Initial Backbone Model Size: 32.47 MB
Modified Backbone Model Size: 28.90 MB

Testing: Total Loss: 33.3912, Cross Entropy: 1.462638, Accuracy: 0.2042, Precision: 0.0417, Recall: 0.2042, F1 Score: 0.0693, AUC: 0.4248,

In [53]:
import onnxruntime as ort

for arch in models_list:
    for mode in ["pytorch", "onnx"]:
        infer(arch, "QUANTIZATION", mode)

mobilenetv2
Initial Backbone Model Size: 14.24 MB
Modified Backbone Model Size: 9.78 MB

Testing mobilenetv2 in pytorch mode
Running inference for fp32
None
Testing: Total Loss: 34.8684, Cross Entropy: 1.362514, Accuracy: 0.3873, Precision: 0.1500, Recall: 0.3873, F1 Score: 0.2163, AUC: 0.5000, R2 Score: -23.814954, MAE: 10.4203, MSE: 113.0487
Avg Latency (ms): 3.75, Avg Memory Before (MB): 0.00, Avg Memory After (MB): 0.00, Avg Max Memory (MB): 0.00
Running inference for fp16
None
Testing: Total Loss: 34.8684, Cross Entropy: 1.362514, Accuracy: 0.3873, Precision: 0.1500, Recall: 0.3873, F1 Score: 0.2163, AUC: 0.5000, R2 Score: -23.814954, MAE: 10.4203, MSE: 113.0487
Avg Latency (ms): 3.07, Avg Memory Before (MB): 0.00, Avg Memory After (MB): 0.00, Avg Max Memory (MB): 0.00
Running inference for int8
None
Testing: Total Loss: 34.8684, Cross Entropy: 1.362514, Accuracy: 0.3873, Precision: 0.1500, Recall: 0.3873, F1 Score: 0.2163, AUC: 0.5000, R2 Score: -23.814954, MAE: 10.4203, MSE: 113

AttributeError: 'NoneType' object has no attribute 'get_inputs'

In [58]:
!pip install onnxruntime-tools
!python -m onnxruntime.tools.check_model /home/sebastian-cruz6/cp-anemia-detection/notebooks/weights/onnx/model_best_accuracy_mobilenetv2_QUANTIZATION.onnx

/home/sebastian-cruz6/cp-anemia-detection/cawt/bin/python: No module named onnxruntime.tools.check_model


In [64]:
import onnx

# Load ONNX model
try:
    onnx_model = onnx.load("/home/sebastian-cruz6/cp-anemia-detection/notebooks/weights/onnx/model_best_accuracy_mobilenetv2_QUANTIZATION.onnx")
    onnx.checker.check_model(onnx_model)
    print("✅ ONNX model is valid!")
except onnx.checker.ValidationError as e:
    print(f"❌ ONNX model validation failed: {e}")

✅ ONNX model is valid!
